# STEP 3 — Exploratory Data Analysis (EDA)
**Project:** AI Chatbot for Mental Health (MSc IT - AI Final Year Project)  
**Standard:** Research-grade, reproducible dataset exploratory analysis without raw data mutation.

## 1. Imports

In [ ]:
# pyright: reportMissingImports=false
import os
import sys
import json
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Handle display import for IDE linters and runtime environments
try:
    from IPython.display import display
except ImportError:
    display = print

# Add project root and src directory to python search path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
src_path = os.path.join(project_root, 'src')
for path in [project_root, src_path]:
    if path not in sys.path:
        sys.path.insert(0, path)

try:
    from src.data.loader import DatasetLoader
except ImportError:
    from data.loader import DatasetLoader

## 2. Configuration Loading

In [ ]:
config_path = '../configs/dataset.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

text_col = config['schema']['text_column']
label_col = config['schema']['label_column']
print(f"Config loaded successfully. Text column: '{text_col}', Label column: '{label_col}'")

## 3. Dataset Loading

In [ ]:
loader = DatasetLoader(config_path=config_path)
df = loader.load_dataset(file_path='../' + config['dataset']['path'])
print(f"Loaded dataset shape: {df.shape}")

## 4. Dataset Overview

In [ ]:
df.info(verbose=False)
display(df.head(3))

## 5. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
total_missing = missing.sum()
print(f"Total missing cells across dataset: {total_missing}")
if total_missing == 0:
    print("✅ 100% complete dataset with ZERO missing values.")

## 6. Duplicate Analysis

In [ ]:
exact_dups = df.duplicated().sum()
text_dups = df[text_col].duplicated().sum()
print(f"Exact row duplicates: {exact_dups}")
print(f"Text column duplicates: {text_dups} ({text_dups/len(df)*100:.2f}%)")

## 7. Target/Class Distribution

In [ ]:
class_counts = df[label_col].value_counts()
class_pcts = df[label_col].value_counts(normalize=True) * 100
print("Class Distribution:")
for cls, count in class_counts.items():
    print(f"  Class {cls}: {count} ({class_pcts[cls]:.2f}%)")

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=label_col, palette=['#4c72b0', '#c44e52'])
plt.title('Target Class Distribution (0: Non-Stress, 1: Stress)')
plt.show()

## 8. Text Length Analysis

In [ ]:
df['word_count'] = df[text_col].astype(str).str.split().str.len()
df['char_count'] = df[text_col].astype(str).str.len()

print("Word Count Summary:")
print(df['word_count'].describe(percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]))

## 9. Word Frequency Analysis (Raw Text)

In [ ]:
all_words = [word.lower() for text in df[text_col] for word in str(text).split()]
word_freq = Counter(all_words)
print("Top 20 raw unigrams (including stopwords):")
for word, cnt in word_freq.most_common(20):
    print(f"  {word}: {cnt}")

## 10. N-Gram Analysis

In [ ]:
def get_bigrams(texts):
    bigrams = []
    for text in texts:
        tokens = str(text).lower().split()
        for i in range(len(tokens) - 1):
            bigrams.append(tokens[i] + ' ' + tokens[i+1])
    return Counter(bigrams)

bigram_freq = get_bigrams(df[text_col])
print("Top 10 raw bigrams:")
for bg, cnt in bigram_freq.most_common(10):
    print(f"  {bg}: {cnt}")

## 11. Class-Wise Text Analysis

In [ ]:
display(df.groupby(label_col)['word_count'].agg(['mean', 'median', 'min', 'max', 'std']))

## 12. Outlier Analysis

In [ ]:
q1 = df['word_count'].quantile(0.25)
q3 = df['word_count'].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
outliers = df[df['word_count'] > upper_bound]
print(f"IQR Upper Bound: {upper_bound} words")
print(f"Outlier records (> {upper_bound:.1f} words): {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")

## 13. Key EDA Findings

### Observed Findings:
1. **Dataset Size**: 3,553 total records across 116 features.
2. **Class Balance**: 1,857 Stress (52.27%) vs 1,696 Non-Stress (47.73%). The dataset is well balanced (ratio 1.09).
3. **Completeness**: 0 missing values across all columns.
4. **Text Length**: Mean word count is 84.12 words (median: 78.0). Range spans from 3 words to 284 words.
5. **Data Leakage Risk**: 21 exact text duplicates and 624 recurring `post_id` segments exist in raw data.

### Recommended Next Steps (Step 4 & Step 5):
1. Apply text-level deduplication prior to stratified train/validation/test splitting.
2. Perform lowercasing, HTML tag stripping, and punctuation normalization during Step 4 NLP cleaning.

## 14. Export Results

In [ ]:
print("EDA notebook execution completed successfully.")